## Part 5

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.special import erfc
from astropy.table import Table

In [2]:
data = Table.read("Planet_Lightcurve.fits")

flux = data['flux']
time = data['time [days]']

x = np.array(time, dtype=float)
y = np.array(flux, dtype=float)

y_err = 0.003

FileNotFoundError: [Errno 2] No such file or directory: 'Planet_Lightcurve.fits'

In [ ]:
def line(x, m, b):
    return m*x+b

def curve(x, a, p, ph):
    return (a*np.sin(p*(x-ph)))+1

# fit line 
ichoose = np.where((np.asarray(x) <= -0.1))[0]
ichoose = np.append(ichoose, np.where((np.asarray(x) >= 0.1))[0])

params, pcov = curve_fit(line, x[ichoose], y[ichoose], sigma=y_err, p0=[.038, 0.985])
m_fit = params[0]
b_fit = params[1]

line_fit = line(x, m_fit, b_fit)
new_y = y / line_fit

# fit oscillation
ichoose2 = np.where((np.asarray(x) <= -0.05))[0]
ichoose2 = np.append(ichoose, np.where((np.asarray(x) >= 0.05))[0])

params2, pcov2 = curve_fit(curve, x[ichoose2], new_y[ichoose2], sigma=y_err, p0=[0.006, 50, 1.5])
a_fit = params2[0]
p_fit = params2[1]
ph_fit = params2[2]

curve_fit_vals = curve(x, a_fit, p_fit, ph_fit)

# cleaned data 
y_best = new_y / curve_fit_vals

In [ ]:
#  noise level of the cleaned lightcurve? I think it works
ichoose3 = np.where((np.asarray(x) <= -0.05))[0]
ichoose3 = np.append(ichoose3, np.where((np.asarray(x) >= 0.05))[0])

flux_oot = y_best[ichoose3]

sigma_clean = np.std(flux_oot)
mean_oot = np.mean(flux_oot)

print(f'mean out-of-transit flux: {mean_oot:.6f}')
print(f'std dev of cleaned flux:  {sigma_clean:.6f}')

In [ ]:
# a 10% solar flare 
flare_flux = mean_oot * 1.10
flare_deviation = flare_flux - mean_oot
N_sigma = flare_deviation / sigma_clean

print(f'flare value:       {flare_flux:.6f}')
print(f'flare deviation:   {flare_deviation:.6f}')
print(f'N-sigma:           {N_sigma:.2f}')

In [ ]:
# inject a single 10% outlier 
flare_idx = np.argmin(np.abs(x - 0.15))  # pick a point at t=0.15 days

x_flare = x.copy()
y_flare = y_best.copy()
y_flare[flare_idx] = flare_flux

print(f'flare injected at t = {x_flare[flare_idx]:.4f} days')

In [ ]:
plt.errorbar(x_flare, y_flare, yerr=y_err, fmt='.', alpha=0.5, label='cleaned flux + flare')
plt.plot(x_flare[flare_idx], y_flare[flare_idx], 'r*', ms=14, label=f'solar flare ({N_sigma:.1f}σ)')
plt.axhline(mean_oot, color='gray', ls='--', lw=1, label='mean baseline')
plt.xlabel('Time [days]')
plt.ylabel('Flux')
plt.title('Cleaned Lightcurve with Injected Solar Flare')
plt.legend()
plt.tight_layout()
plt.savefig('flare_lightcurve.pdf')
plt.show()

In [ ]:
# Chauvenet's criterion

N_oot = len(flux_oot)
expected = N_oot * erfc(N_sigma / np.sqrt(2))

print(f'N (out-of-transit points): {N_oot}')
print(f'N-sigma:                   {N_sigma:.2f}')
print(f'expected outliers (N*P):   {expected:.2e}')
print()
if expected < 0.5:
    print('expected < 0.5 → significant by Chauvenet criterion')
else:
    print('expected >= 0.5 → not significant by Chauvenet criterion')